In [3]:
import pandas as pd


df=pd.read_csv('../data/Bitcoin Historical Data.csv')

print(df.head())
df.info()


         Date     Price      Open      High       Low     Vol. Change %
0  24-03-2024  67,211.9  64,036.5  67,587.8  63,812.9   65.59K    4.96%
1  23-03-2024  64,037.8  63,785.6  65,972.4  63,074.9   35.11K    0.40%
2  22-03-2024  63,785.5  65,501.5  66,633.3  62,328.3   72.43K   -2.62%
3  21-03-2024  65,503.8  67,860.0  68,161.7  64,616.1   75.26K   -3.46%
4  20-03-2024  67,854.0  62,046.8  68,029.5  60,850.9  133.53K    9.35%
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Date      4999 non-null   object
 1   Price     4999 non-null   object
 2   Open      4999 non-null   object
 3   High      4999 non-null   object
 4   Low       4999 non-null   object
 5   Vol.      4993 non-null   object
 6   Change %  4999 non-null   object
dtypes: object(7)
memory usage: 273.5+ KB


In [6]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

for col in ['Price', 'Open', 'High', 'Low']:
    df[col] = df[col].astype(str).str.replace(',', '').astype(float)

df['Change %'] = df['Change %'].astype(str).str.replace('%', '').astype(float)

def convert_vol(value):
    value = str(value).upper()
    if 'K' in value:
        return float(value.replace('K', '')) * 1000
    if 'M' in value:
        return float(value.replace('M', '')) * 1000000
    try:
        return float(value)
    except:
        return 0.0 
df['Vol.'] = df['Vol.'].apply(convert_vol)

In [7]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      4999 non-null   datetime64[ns]
 1   Price     4999 non-null   float64       
 2   Open      4999 non-null   float64       
 3   High      4999 non-null   float64       
 4   Low       4999 non-null   float64       
 5   Vol.      4993 non-null   float64       
 6   Change %  4999 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 273.5 KB


In [9]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# 1. Select the features for the model
features = ['Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
data_values = df[features].values

# 2. Split data (80% Train, 20% Test) - NO SHUFFLING
train_idx = int(len(data_values) * 0.8)
train_data = data_values[:train_idx]
test_data = data_values[train_idx:]

# 3. Scale the data to (0, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# 4. Create the 60-day sequences
def create_sequences(data, window=60, horizon=1):
    X, y = [], []
    for i in range(len(data) - window - horizon + 1):
        X.append(data[i : i + window])
        y.append(data[i + window + horizon - 1, 0]) # Target is the 'Price'
    return np.array(X), np.array(y)

# Prepare 1-Day Forecast data
X_train, y_train = create_sequences(train_scaled, window=60, horizon=1)
X_test, y_test = create_sequences(test_scaled, window=60, horizon=1)

print(f"Foundation Ready! Training shape: {X_train.shape}")

Foundation Ready! Training shape: (3939, 60, 6)


In [ ]:
import numpy as np
import os


os.makedirs('../data', exist_ok=True)


np.save('../data/X_train_1d.npy', X_train)
np.save('../data/y_train_1d.npy', y_train)
np.save('../data/X_test_1d.npy', X_test)
np.save('../data/y_test_1d.npy', y_test)

print("Success! Check your '../data/' folder now. You should see four .npy files.")


Success! Check your '../data/' folder now. You should see four .npy files.


In [ ]:

df.to_csv('../data/bitcoin_processed.csv', index=False)


df_raw = pd.read_csv('../data/Bitcoin Historical Data.csv')
df_raw.to_csv('../data/bitcoin_raw.csv', index=False)